# ARSET Surface Urban Heat Island (SUHI) index

Reproduction of the heat-island index from NASA ARSET (2022):
[Satellite Remote Sensing for Measuring Urban Heat Islands and Constructing Heat Vulnerability Indices](https://www.earthdata.nasa.gov/learn/trainings/satellite-remote-sensing-measuring-urban-heat-islands-constructing-heat).

Part 1 defines **SUHI intensity** as the difference between mean urban and mean rural land surface temperature:

$$\Delta T_{u-r} = T_u - T_r$$

This notebook ports the official `Landsat_LST_SUHI.js` workflow (Washington, D.C. demo) to Python / Google Earth Engine:

1. Landsat 8 & 9 Collection 2 Level-2 surface temperature (`ST_B10`)
2. Cloud / cloud-shadow mask from `QA_PIXEL` bits 3 and 4
3. USGS scale factors to convert DNs to °C
4. Summer composite (day of year 182–243, July–August)
5. Urban vs rural statistics and the SUHI index

Hand-drawn urban/rural polygons from the GEE Code Editor are replaced with **GHSL Settlement Model** classes so the same city (or Madrid) can be rerun without redrawing. The formula is unchanged.

A later section applies the same index to **ECOSTRESS** LST.

**Cite:** NASA Applied Remote Sensing Training (ARSET) program. McCartney, S. & Mehta, A. (2022).

In [ ]:
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt

try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize(project="ee-my-usenvdi")

## 1. Study area and time window

Change `CITY` to `Madrid` (or add another entry) to keep the same method on a different city.
The ARSET lab used July–August (`DATE_RANGE` = day of year 182–243) because SUHI is typically strongest in summer.

In [ ]:
CITIES = {
    # Official ARSET Part 1 demo city
    "Washington_DC": {"lon": -77.0369, "lat": 38.9072, "buffer_m": 40000, "zoom": 10},
    # Matches the Madrid AOI used in Olas_de_calor.ipynb
    "Madrid": {"lon": -3.7038, "lat": 40.4168, "buffer_m": 40000, "zoom": 10},
}

CITY = "Washington_DC"
cfg = CITIES[CITY]

aoi = ee.Geometry.Point([cfg["lon"], cfg["lat"]]).buffer(cfg["buffer_m"]).bounds()

# ARSET defaults: summer (1 Jul – 31 Aug), Landsat 8 era through the 2022 training
START_YEAR = 2013
END_YEAR = 2022
DOY_START = 182  # 1 July
DOY_END = 243    # 31 August
CLOUD_COVER_MAX = 20

print(f"City: {CITY}")
print(f"Years: {START_YEAR}–{END_YEAR}, DOY {DOY_START}–{DOY_END}, cloud cover < {CLOUD_COVER_MAX}%")

## 2. Landsat surface temperature (ARSET workflow)

Functions follow the training script:

- Mask pixels where `QA_PIXEL` bit 3 (cloud) or bit 4 (cloud shadow) is set
- Convert `ST_B10` with USGS Collection 2 scale factors: `DN × 0.00341802 + 149` (Kelvin), then `− 273.15` (°C)
- Keep scenes with `< 20%` cloud cover
- Mean composite over the filtered collection

In [ ]:
def cloud_mask(image):
    qa = image.select("QA_PIXEL")
    mask = qa.bitwiseAnd(1 << 3).Or(qa.bitwiseAnd(1 << 4))
    return image.updateMask(mask.Not())


def apply_scale_factors(image):
    thermal = (
        image.select("ST_B10")
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename("ST_C")
    )
    return image.addBands(thermal, None, True)


def load_landsat_st(collection_id):
    return (
        ee.ImageCollection(collection_id)
        .select(["ST_B10", "QA_PIXEL"])
        .filterBounds(aoi)
        .filter(ee.Filter.dayOfYear(DOY_START, DOY_END))
        .filter(ee.Filter.calendarRange(START_YEAR, END_YEAR, "year"))
        .filter(ee.Filter.lt("CLOUD_COVER", CLOUD_COVER_MAX))
        .map(cloud_mask)
        .map(apply_scale_factors)
    )


landsat = load_landsat_st("LANDSAT/LC08/C02/T1_L2").merge(
    load_landsat_st("LANDSAT/LC09/C02/T1_L2")
)

n_images = landsat.size().getInfo()
print(f"Landsat 8/9 scenes after filters: {n_images}")

mean_st = landsat.select("ST_C").mean().clip(aoi)
print("Mean summer surface temperature composite is ready.")

## 3. Urban and rural reference areas

The ARSET script asked you to draw `Urban` and `Rural` polygons. Here the same contrast is built from GHSL Degree of Urbanisation (2020):

| Role | GHSL `smod_code` |
| --- | --- |
| Urban | 30 urban centre, 23 dense urban cluster, 22 semi-dense urban cluster |
| Rural | 13 rural cluster, 12 low-density rural, 11 very low-density rural |
| Excluded | 10 water, 21 suburban/peri-urban |

Suburban pixels are left out so the rural mean is not pulled toward the city edge. Water is masked from both classes.

In [ ]:
smod = ee.Image("JRC/GHSL/P2023A/GHS_SMOD_V2-0/2020").select("smod_code").clip(aoi)

urban_mask = smod.eq(30).Or(smod.eq(23)).Or(smod.eq(22))
rural_mask = smod.eq(13).Or(smod.eq(12)).Or(smod.eq(11))

mean_st_land = mean_st.updateMask(smod.neq(10))
urban_st = mean_st_land.updateMask(urban_mask)
rural_st = mean_st_land.updateMask(rural_mask)

print("Urban / rural masks built from GHSL 2020.")

## 4. SUHI index

ARSET: find mean ST under urban and rural features, then

`SUHI intensity = ST_mean [Urban] − ST_mean [Rural]`

The pixel map is the same idea at every location: `LST − T_r`. Positive values are hotter than the rural reference.

In [ ]:
def zone_stats(image, geom, scale=30):
    reducer = (
        ee.Reducer.mean()
        .combine(ee.Reducer.minMax(), sharedInputs=True)
        .combine(ee.Reducer.count(), sharedInputs=True)
        .combine(ee.Reducer.stdDev(), sharedInputs=True)
    )
    return image.reduceRegion(
        reducer=reducer,
        geometry=geom,
        scale=scale,
        maxPixels=1e9,
        bestEffort=True,
    )


urban_info = zone_stats(urban_st, aoi).getInfo()
rural_info = zone_stats(rural_st, aoi).getInfo()

t_urban = urban_info["ST_C_mean"]
t_rural = rural_info["ST_C_mean"]
suhi_intensity = t_urban - t_rural

summary = pd.DataFrame(
    {
        "zone": ["Urban", "Rural"],
        "mean_ST_C": [t_urban, t_rural],
        "min_ST_C": [urban_info["ST_C_min"], rural_info["ST_C_min"]],
        "max_ST_C": [urban_info["ST_C_max"], rural_info["ST_C_max"]],
        "std_ST_C": [urban_info["ST_C_stdDev"], rural_info["ST_C_stdDev"]],
        "n_pixels": [urban_info["ST_C_count"], rural_info["ST_C_count"]],
    }
)

print(summary.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))
print()
print(f"SUHI intensity  ΔT_u−r  =  {t_urban:.2f} − {t_rural:.2f}  =  {suhi_intensity:.2f} °C")
print("Positive values mean the urban area is warmer than the rural reference.")

t_rural_img = ee.Image.constant(t_rural)
suhi_map = mean_st_land.subtract(t_rural_img).rename("SUHI")

In [ ]:
samples = geemap.ee_to_df(
    mean_st_land.sample(region=aoi, scale=90, numPixels=4000, geometries=False)
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(samples["ST_C"], bins=40, color="#c0392b", alpha=0.85, edgecolor="white")
ax.axvline(t_urban, color="#1a1a1a", linestyle="--", linewidth=1.4, label=f"Urban mean {t_urban:.1f} °C")
ax.axvline(t_rural, color="#1f6f8b", linestyle="--", linewidth=1.4, label=f"Rural mean {t_rural:.1f} °C")
ax.set_xlabel("Mean summer surface temperature (°C)")
ax.set_ylabel("Sampled pixels")
ax.set_title(f"Landsat ST histogram — {CITY.replace('_', ' ')}")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Map: mean LST and SUHI

Toggle layers in the map. The SUHI layer is `LST − T_rural`; the printed `ΔT_u−r` above is the city-wide index from the training.

In [ ]:
lst_vis = {"min": 28, "max": 47, "palette": ["blue", "white", "red"]}
suhi_vis = {
    "min": -4,
    "max": 8,
    "palette": ["#2166ac", "#67a9cf", "#d1e5f0", "#f7f7f7", "#fddbc7", "#ef8a62", "#b2182b"],
}
urban_vis = {"min": 0, "max": 1, "palette": ["#ff0000"]}
rural_vis = {"min": 0, "max": 1, "palette": ["#2ecc71"]}

Map = geemap.Map()
Map.centerObject(aoi, cfg["zoom"])
Map.addLayer(mean_st_land, lst_vis, f"Mean summer LST (°C) — {CITY}")
Map.addLayer(suhi_map, suhi_vis, f"SUHI (°C) = LST − Trural")
Map.addLayer(urban_mask.selfMask(), urban_vis, "Urban (GHSL)", False)
Map.addLayer(rural_mask.selfMask(), rural_vis, "Rural (GHSL)", False)
Map.addLayer(ee.Image().paint(ee.FeatureCollection([ee.Feature(aoi)]), 0, 2), {"palette": "black"}, "AOI")
Map.add_colorbar(lst_vis, label="LST (°C)", layer_name="Mean summer LST")
Map.add_colorbar(suhi_vis, label="SUHI (°C)", layer_name="SUHI")
Map

## 6. Same index with ECOSTRESS

The ARSET lab used Landsat (daytime, ~10:30 local) and Aqua MODIS (day and night). This project already uses ECOSTRESS Collection 2, so the same `T_u − T_r` index is computed on ECOSTRESS LST with the **same urban/rural masks**.

ECOSTRESS overpasses at many hours; the filter keeps daytime scenes (10:00–16:00 UTC offset converted from local solar-ish hours via longitude) so the number is comparable to Landsat. Adjust `ECO_START` / `ECO_END` if the collection is sparse over the city.

In [ ]:
ECO_START = f"{max(START_YEAR, 2019)}-07-01"
ECO_END = f"{END_YEAR}-09-01"


def prep_ecostress(image):
    lst = image.select("LST").multiply(0.02).subtract(273.15).rename("ST_C")
    qa_mask = image.select("QC").bitwiseAnd(3).lt(2)
    cloud_ok = image.select("cloud").eq(0)
    utc_hour = ee.Number(ee.Date(image.get("system:time_start")).get("hour"))
    local_hour = utc_hour.add(cfg["lon"] / 15.0).add(24).mod(24)
    return (
        lst.updateMask(qa_mask)
        .updateMask(cloud_ok)
        .copyProperties(image, ["system:time_start"])
        .set("local_hour", local_hour)
    )


eco_col = (
    ee.ImageCollection("NASA/ECOSTRESS/L2T_LSTE/V2")
    .filterBounds(aoi.centroid())
    .filterDate(ECO_START, ECO_END)
    .filter(ee.Filter.dayOfYear(DOY_START, DOY_END))
    .map(prep_ecostress)
    .filter(ee.Filter.gte("local_hour", 10))
    .filter(ee.Filter.lt("local_hour", 16))
)

n_eco = eco_col.size().getInfo()
print(f"ECOSTRESS daytime summer scenes: {n_eco}")

if n_eco == 0:
    print("No ECOSTRESS scenes for this city/window. The Landsat SUHI index above is still valid.")
    Map_eco = None
else:
    eco_mean = eco_col.mean().clip(aoi).updateMask(smod.neq(10))
    eco_urban = zone_stats(eco_mean.updateMask(urban_mask), aoi, scale=70).getInfo()
    eco_rural = zone_stats(eco_mean.updateMask(rural_mask), aoi, scale=70).getInfo()
    eco_suhi = eco_urban["ST_C_mean"] - eco_rural["ST_C_mean"]
    eco_suhi_map = eco_mean.subtract(eco_rural["ST_C_mean"]).rename("SUHI")

    print(
        f"ECOSTRESS SUHI  ΔT_u−r  =  {eco_urban['ST_C_mean']:.2f} − {eco_rural['ST_C_mean']:.2f}  =  {eco_suhi:.2f} °C"
    )

    Map_eco = geemap.Map()
    Map_eco.centerObject(aoi, cfg["zoom"])
    Map_eco.addLayer(eco_mean, lst_vis, f"ECOSTRESS mean LST (°C) — {CITY}")
    Map_eco.addLayer(eco_suhi_map, suhi_vis, "ECOSTRESS SUHI (°C)")
    Map_eco.add_colorbar(suhi_vis, label="SUHI (°C)", layer_name="ECOSTRESS SUHI")

Map_eco


## Notes

- **This is the heat island index from Part 1**, not the Heat Vulnerability Index (HVI) from Parts 2–3. HVI combines LST with socioeconomic indicators (age, poverty, vegetation, etc.) using normalization, equal weights, or PCA. The Part 3 demo data are on the [training page](https://www.earthdata.nasa.gov/learn/trainings/satellite-remote-sensing-measuring-urban-heat-islands-constructing-heat).
- Official GEE scripts from the slides:
  - [Landsat LST time series](https://code.earthengine.google.com/2eae305edb229ae171391b341094fa81)
  - [Landsat SUHI (Washington, D.C.)](https://code.earthengine.google.com/8f8a363aa18fa9d16c1fe84991aa4154)
  - [MODIS day/night SUHI](https://code.earthengine.google.com/63c37316806efa35321f7e8651429bb2)
- Landsat SUHI is **daytime only**. Nighttime intensity needs MODIS or ECOSTRESS.
- To match the original lab exactly, draw urban and rural polygons in the map and replace the GHSL masks with `clip_mean_ST.reduceRegions({collection: Urban, ...})` as in the JavaScript.